# qlora_convert — quantized checkpoint to loadable adapter

Each QLoRA run saves `checkpoint-N/model.safetensors`: 2,431 tensors holding the NF4 base
weights as uint8 and the bf16 adapter together, about 2.06 GB. This notebook extracts the 304
adapter tensors and writes `pytorch_lora_weights.safetensors` in the form
`pipe.transformer.load_lora_adapter(..., prefix="transformer")` accepts, so the QLoRA arm
loads through the same call the LoRA arm uses.

Simpler than `lora_convert`, which had to rebuild the transformer on CPU and run
`get_peft_model_state_dict` to pull the adapter out of a fused FSDP state dict. Without FSDP
the adapter tensors are already separate entries under PEFT names, so extraction is a key
filter and rename:

    context_refiner.0.attn.to_k.lora_A.default.weight
        -> transformer.context_refiner.0.attn.to_k.lora_A.weight

No model is constructed, no weights are dequantized, and peft is never invoked, so the
torchao dispatcher issue that `lora_convert` works around does not arise. `safe_open` reads
the header without loading tensor data, so only the 304 adapter tensors are ever read.

## 1. Drive and checkpoint inventory

In [ ]:
import os, json
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
EXP   = f"{DRIVE}/lora/experiments"
RANKS = [8]                       # r4 / r16 only if the rank sweep is repeated under quantization
STEPS = [500, 1000, 1500, 2000]   # every checkpoint written by qlora_train

found = []
for r in RANKS:
    cfg = f"{EXP}/qlora_r{r}/qlora_r{r}.yml"
    assert os.path.exists(cfg), f"missing config: {cfg}"
    for s in STEPS:
        ck = f"{EXP}/qlora_r{r}/checkpoint-{s}/model.safetensors"
        if os.path.exists(ck):
            found.append((r, s))
            print(f"r{r} step {s:>5}: {os.path.getsize(ck)/1e9:.2f} GB")
        else:
            print(f"r{r} step {s:>5}: absent")
assert found, "no checkpoints found"

Mounted at /content/drive
r8 step   500: 2.06 GB
r8 step  1000: 2.06 GB
r8 step  1500: 2.06 GB
r8 step  2000: 2.06 GB


## 2. Extract

The adapter is written in PEFT's active-adapter form, so every key carries a `.default.`
infix and no module prefix. `load_lora_adapter` expects diffusers' form: a `transformer.`
prefix and no adapter name. The rename is exact and reversible; tensor values are copied
unchanged.

`expected` is `38 * 16800 * rank`, the parameter count the training run reported: 38 attention
blocks across the main stack and the three refiners, four projections each, with GQA giving
to_k and to_v an output width of 840 against 2520 for to_q and to_out.0.

In [ ]:
from safetensors import safe_open
from safetensors.torch import save_file

def extract(rank, step):
    src  = f"{EXP}/qlora_r{rank}/checkpoint-{step}/model.safetensors"
    save = f"{EXP}/qlora_r{rank}/checkpoint-{step}/transformer_lora"
    os.makedirs(save, exist_ok=True)
    out  = f"{save}/pytorch_lora_weights.safetensors"

    adapter, base_dtypes = {}, set()
    with safe_open(src, framework="pt") as f:
        keys = list(f.keys())
        for k in keys:
            if ".lora_A" not in k and ".lora_B" not in k:
                continue
            assert ".default." in k, f"unexpected adapter key form: {k}"
            adapter["transformer." + k.replace(".default.", ".")] = f.get_tensor(k)
        for k in keys[:200]:
            if "base_layer.weight" in k:
                base_dtypes.add(str(f.get_slice(k).get_dtype()))

    n_params = sum(v.numel() for v in adapter.values())
    expected = 38 * 16800 * rank
    assert len(adapter) == 304, f"r{rank} step {step}: {len(adapter)} adapter tensors, expected 304"
    assert n_params == expected, f"r{rank} step {step}: {n_params:,} params, expected {expected:,}"
    assert all(v.dtype.is_floating_point for v in adapter.values()), "adapter tensor is not floating point"

    save_file(adapter, out)
    print(f"r{rank} step {step:>5}: {len(adapter)} tensors, {n_params/1e6:.3f} M params, "
          f"{os.path.getsize(out)/1e6:.1f} MB, base {sorted(base_dtypes)} -> {out}")
    return out

written = [extract(r, s) for r, s in found]

r8 step   500: 304 tensors, 5.107 M params, 10.2 MB, base ['F32', 'U8'] -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/qlora_r8/checkpoint-500/transformer_lora/pytorch_lora_weights.safetensors
r8 step  1000: 304 tensors, 5.107 M params, 10.2 MB, base ['F32', 'U8'] -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/qlora_r8/checkpoint-1000/transformer_lora/pytorch_lora_weights.safetensors
r8 step  1500: 304 tensors, 5.107 M params, 10.2 MB, base ['F32', 'U8'] -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/qlora_r8/checkpoint-1500/transformer_lora/pytorch_lora_weights.safetensors
r8 step  2000: 304 tensors, 5.107 M params, 10.2 MB, base ['F32', 'U8'] -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/qlora_r8/checkpoint-2000/transformer_lora/pytorch_lora_weights.safetensors


## 3. Verify

Reads each written adapter back and checks the properties the inference path depends on: the
key form `load_lora_adapter` matches, one A and one B tensor per adapted module, and the rank
implied by the tensor shapes. `lora_A` is `[r, in_features]` and `lora_B` is `[out_features, r]`,
so the shared dimension is the rank the run was configured with.

In [ ]:
from safetensors.torch import load_file
import collections

for path in written:
    sd = load_file(path)
    a = {k for k in sd if ".lora_A." in k}
    b = {k for k in sd if ".lora_B." in k}
    ranks = {sd[k].shape[0] for k in a} | {sd[k].shape[1] for k in b}
    prefixed = all(k.startswith("transformer.") for k in sd)
    residual = [k for k in sd if ".default." in k]
    modules = {k.rsplit(".lora_", 1)[0] for k in sd}

    print(f"\n{path.split('/')[-3]} {path.split('/')[-2]}")
    print(f"  tensors        : {len(sd)}   A {len(a)} / B {len(b)}")
    print(f"  adapted modules: {len(modules)}")
    print(f"  implied rank   : {sorted(ranks)}")
    print(f"  dtype          : {sorted({str(v.dtype) for v in sd.values()})}")
    print(f"  transformer.   : {prefixed}   leftover '.default.': {len(residual)}")
    assert len(a) == len(b) == 152, "adapter is not 152 A/B pairs"
    assert len(ranks) == 1, f"inconsistent rank across tensors: {sorted(ranks)}"
    assert prefixed and not residual, "key form will not load via load_lora_adapter"

print("\nexample key:", next(iter(sd)))


checkpoint-500 transformer_lora
  tensors        : 304   A 152 / B 152
  adapted modules: 152
  implied rank   : [8]
  dtype          : ['torch.bfloat16']
  transformer.   : True   leftover '.default.': 0

checkpoint-1000 transformer_lora
  tensors        : 304   A 152 / B 152
  adapted modules: 152
  implied rank   : [8]
  dtype          : ['torch.bfloat16']
  transformer.   : True   leftover '.default.': 0

checkpoint-1500 transformer_lora
  tensors        : 304   A 152 / B 152
  adapted modules: 152
  implied rank   : [8]
  dtype          : ['torch.bfloat16']
  transformer.   : True   leftover '.default.': 0

checkpoint-2000 transformer_lora
  tensors        : 304   A 152 / B 152
  adapted modules: 152
  implied rank   : [8]
  dtype          : ['torch.bfloat16']
  transformer.   : True   leftover '.default.': 0

example key: transformer.context_refiner.0.attn.to_k.lora_A.weight


## 4. Load test

Loads one adapter through the exact call the evaluation notebooks use, against a
quantized transformer, and confirms peft reports the adapter attached. Weights only; no
image is generated.

The base is loaded NF4 to match training. An adapter trained against a quantized base is
evaluated against the same quantized base, so the arm measures adaptation rather than a
train/test precision mismatch.

In [ ]:
LOAD_TEST = True

if LOAD_TEST:
    import torch, gc, sys, subprocess
    REPO   = "/content/OmniGen2"
    COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"
    if not os.path.isdir(REPO):
        subprocess.run(["git","clone",
                        "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
        subprocess.run(["git","-C",REPO,"checkout","--quiet",COMMIT], check=True)
    _head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"],
                           capture_output=True, text=True).stdout.strip()
    assert _head == COMMIT, f"repo at {_head[:7]}, expected {COMMIT[:7]}"
    # DiffusersBnBConfig.post_init() reads importlib.metadata.version("bitsandbytes")
    subprocess.run([sys.executable,"-m","pip","install","-q","bitsandbytes==0.50.0"], check=True)
    sys.path.insert(0, REPO)
    from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel
    from diffusers import BitsAndBytesConfig as DiffusersBnBConfig

    qc = DiffusersBnBConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.bfloat16,
                            bnb_4bit_use_double_quant=True)
    tr = OmniGen2Transformer2DModel.from_pretrained(
        "OmniGen2/OmniGen2", subfolder="transformer",
        quantization_config=qc, torch_dtype=torch.bfloat16)

    d = os.path.dirname(written[-1])
    tr.load_lora_adapter(d, weight_name="pytorch_lora_weights.safetensors", prefix="transformer")
    n_lora = sum(1 for n, _ in tr.named_parameters() if "lora_" in n)
    print(f"loaded {d}")
    print(f"  lora parameter tensors on the model: {n_lora}   (expect 304)")
    assert n_lora == 304, "adapter did not attach"
    del tr; gc.collect(); torch.cuda.empty_cache()
else:
    print("LOAD_TEST=False - skipped")

/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF

config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

(…)ion_pytorch_model.safetensors.index.json:   0%|          | 0.00/53.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/lora/experiments/qlora_r8/checkpoint-2000/transformer_lora
  lora parameter tensors on the model: 304   (expect 304)
